# Tracera — Data Sources Index

The data layer behind Tracera: **20 vetted sources** for farm decisions, each with its own
notebook that documents the source *and* contains a working API connection.

Every point/field query uses one shared location — **Story County, Iowa** (`42.05, -93.50`,
a real corn/soybean field) — so results line up across sources. See **`README.md`** for the
full write-up and setup.

**How to use:** open any source notebook and *Run All*. This index gives the catalog, the
credential checklist, and a **live smoke-test** of the no-key sources.

## Catalog

| Category | Source | Notebook | Key |
|---|---|---|:--:|
| Crops/yield | NASS QuickStats | `Prototype_quick_Stats.ipynb` | have it |
| Crops/econ | ERS ARMS | `ers_arms.ipynb` | DEMO_KEY |
| Soil | SSURGO (Soil Data Access) | `soil_ssurgo.ipynb` | none |
| Soil | ISRIC SoilGrids | `soil_grids.ipynb` | none |
| Weather | gridMET (4 km US) | `gridmet.ipynb` | none |
| Weather | NASA POWER | `nasa_power_api.ipynb` | none |
| Weather | Open-Meteo | `open_meteo_historical.ipynb` | none |
| Weather | NOAA NCEI/CDO | `Prototype_weather_noaa.ipynb` | optional |
| Water/ET | OpenET | `openet.ipynb` | **key** |
| Water | USGS NWIS | `usgs_water.ipynb` | none |
| Water/soil | NRCS SCAN/SNOTEL | `nrcs_scan_snotel.ipynb` | none |
| Vegetation | Sentinel-2 NDVI (Planetary Computer) | `satellite_ndvi.ipynb` | optional |
| Vegetation | Soil moisture + VegScape/CROP-CASMA | `vegscape_cropcasma.ipynb` | none |
| Land use | Cropland Data Layer | `cropland_cdl.ipynb` | none |
| Risk | US Drought Monitor | `drought_monitor.ipynb` | none |
| Risk | RMA Cause of Loss | `rma_crop_loss.ipynb` | none |
| Markets | Grain futures (Yahoo) | `cme.ipynb` | none |
| Markets | AMS Market News | `ams_market_news_prototype.ipynb` | **key** |
| Econ | FRED | `fred_prototype.ipynb` | **key** |
| Terrain | USGS 3DEP elevation | `usgs_elevation.ipynb` | none |

## Credential checklist

Paste any keys you obtain into `.env` (see `.env.example`). Everything not listed needs **no key**.

| Env var | Unlocks | Free signup |
|---|---|---|
| `NASS_API_KEY` | QuickStats | *already set* |
| `OPENET_API_KEY` | OpenET (crop water use) | etdata.org |
| `FRED_API_KEY` | FRED (economics) | fredaccount.stlouisfed.org/apikeys |
| `AMS_API_KEY` | AMS Market News | mymarketnews.ams.usda.gov |
| `ERS_API_KEY` | ERS ARMS (else DEMO_KEY) | api.data.gov/signup |
| `NOAA_CDO_TOKEN` | NOAA station discovery | ncdc.noaa.gov/cdo-web/token |
| `PC_SDK_SUBSCRIPTION_KEY` | Planetary Computer limits | planetarycomputer.microsoft.com |

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


## Live smoke-test — the free / demo stack, one small pull each

In [2]:
import time

def check(fn):
    t0 = time.time()
    try:
        return f"OK  ·  {fn()}", round(time.time() - t0, 1)
    except Exception as e:
        return f"FAIL · {type(e).__name__}: {str(e)[:60]}", round(time.time() - t0, 1)

CDL = "https://nassgeodata.gmu.edu/axis2/services/CDLService"
from pyproj import Transformer
_alb = Transformer.from_crs("EPSG:4326", "EPSG:5070", always_xy=True)

def s_quickstats():
    from data_sources.quickstats import county_yield
    y = county_yield("CORN").dropna(subset=["Value"]).iloc[-1]
    return f"corn yield {int(y.year)} = {y.Value:.0f} bu/ac"

def s_ers():
    d = requests.get("https://api.ers.usda.gov/data/arms/year",
                     params={"api_key": get_key("ERS_API_KEY", required=False) or "DEMO_KEY"}, timeout=30).json()["data"]
    return f"{len(d)} ARMS years ({min(d)}–{max(d)})"

def s_ssurgo():
    q = f"SELECT TOP 1 muname FROM mapunit WHERE mukey IN (SELECT mukey FROM SDA_Get_Mukey_from_intersection_with_WktWgs84('point({LON} {LAT})'))"
    t = requests.post("https://sdmdataaccess.sc.egov.usda.gov/Tabular/post.rest",
                      json={"query": q, "format": "JSON+COLUMNNAME"}, timeout=45).json()["Table"]
    return t[1][0][:38]

def s_cdl():
    import re
    x, y = _alb.transform(LON, LAT)
    r = requests.get(f"{CDL}/GetCDLValue", params={"year": 2023, "x": x, "y": y}, timeout=45)
    return "2023 crop = " + re.search(r'category: "([^"]+)"', r.text).group(1)

def s_drought():
    d = requests.get("https://usdmdataservices.unl.edu/api/CountyStatistics/GetDroughtSeverityStatisticsByAreaPercent",
        params={"aoi": FIPS, "startdate": "1/1/2023", "enddate": "12/31/2023", "statisticsType": "1"},
        headers={"Accept": "application/json"}, timeout=45).json()
    return f"{len(d)} weekly drought records (2023)"

def s_openmeteo():
    from data_sources.weather import open_meteo_daily
    v = open_meteo_daily("2023-07-01", "2023-07-01")["temperature_2m_max"].iloc[0]
    return f"Jul 1 2023 tmax {v:.0f}C"

def s_power():
    r = requests.get("https://power.larc.nasa.gov/api/temporal/daily/point", params={"parameters": "T2M",
        "community": "AG", "latitude": LAT, "longitude": LON, "start": "20230701", "end": "20230701",
        "format": "JSON"}, timeout=45).json()
    return f"Jul 1 2023 T2M {list(r['properties']['parameter']['T2M'].values())[0]}C"

def s_elev():
    from data_sources.elevator import elevation
    return f"{elevation():.0f} m"

def s_water():
    from data_sources.weather import SAMPLE_FARM as _sf  # noqa
    r = requests.get("https://waterservices.usgs.gov/nwis/site/", params={"format": "rdb",
        "bBox": f"{LON-0.4:.3f},{LAT-0.3:.3f},{LON+0.4:.3f},{LAT+0.3:.3f}", "parameterCd": "00060",
        "siteStatus": "active", "hasDataTypeCd": "dv"}, timeout=45).text
    n = len([l for l in r.splitlines() if l and not l.startswith("#")]) - 2
    return f"{max(n,0)} active streamflow gages nearby"

def s_futures():
    from data_sources.futures import grain_futures
    px = grain_futures(start="2024-12-01").dropna()
    return f"corn ${px['Corn'].iloc[-1]:.2f}/bu (latest)"

CHECKS = {
    "NASS QuickStats": s_quickstats, "ERS ARMS (DEMO_KEY)": s_ers,
    "SSURGO soil": s_ssurgo, "Cropland Data Layer": s_cdl,
    "US Drought Monitor": s_drought, "Open-Meteo": s_openmeteo,
    "NASA POWER": s_power, "USGS elevation": s_elev,
    "USGS water": s_water, "Grain futures": s_futures,
}
rows = []
for name, fn in CHECKS.items():
    result, secs = check(fn)
    rows.append({"source": name, "result": result, "sec": secs})
status = pd.DataFrame(rows).set_index("source")
print("Free-stack smoke test (the remaining sources need a key — see checklist):")
status

Free-stack smoke test (the remaining sources need a key — see checklist):


,result,sec
source,,
NASS QuickStats,OK · corn yield 2025 = 204 bu/ac,0.8
ERS ARMS (DEMO_KEY),OK · 29 ARMS years (1996–2024),1.3
SSURGO soil,"OK · Canisteo clay loam, Bemis moraine, 0 t",0.8
Cropland Data Layer,OK · 2023 crop = Soybeans,0.8
US Drought Monitor,OK · 53 weekly drought records (2023),0.7
Open-Meteo,OK · Jul 1 2023 tmax 24C,0.9
NASA POWER,OK · Jul 1 2023 T2M 22.39C,0.9
USGS elevation,OK · 296 m,1.2
USGS water,OK · 5 active streamflow gages nearby,13.3


## How the sources combine

- **Yield benchmark** — QuickStats (county yield) × CDL (was this field that crop?).
- **Water balance** — OpenET (actual ET) − gridMET/Open-Meteo (reference ET & rain) + NASA POWER
  soil moisture + USGS supply.
- **Risk profile** — RMA Cause of Loss × Drought Monitor × SSURGO drainage.
- **Marketing** — futures + AMS cash = basis; FRED + ERS ARMS for cost/margin.
- **Crop health in-season** — Sentinel-2 NDVI × soil moisture × Growing Degree Days.

**Next steps:** add the three keys (OpenET, FRED, AMS) to unlock the full set; then wire the
`data_sources/` helpers into the app backend.